In [ ]:
import pathlib, sys, random
_root  = pathlib.Path('..').resolve()
_tests = pathlib.Path('../tests').resolve()
for _p in [str(_root), str(_tests)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import polars as pl
from polars2svg import Polars2SVG
from IPython.display import HTML, display
from histop_dataframes import makeHistoDf
from timep_dataframes  import makeTimeDf

p2s = Polars2SVG()


def show_row(*items, df=None):
    """Render (label, viz) pairs side-by-side in a single HTML table row."""
    cells = ''.join(
        f'<td style="padding:6px 10px;vertical-align:top;text-align:center">'
        f'<div style="font-size:11px;color:#888;margin-bottom:4px">{label}</div>'
        f'{viz._repr_svg_()}</td>'
        for label, viz in items
    )
    if df is not None:
        display(df)
    display(HTML(f'<table><tr>{cells}</tr></table>'))

---
## Histop — Color Modes
_Source: `test_histop_color.py`_

DataFrame: `cat` ∈ {A, B, C}, `group` ∈ {x, y}, `value` (int), `score` (float).

In [ ]:
df_h = makeHistoDf(n=200, seed=42)
display(df_h.head())

### No color / color same as bin / categorical stacked / numeric spectrum

In [ ]:
wxh = (200, 160)
show_row(
    ('color=None (default)',        p2s.histop(df_h, 'cat', wxh=wxh)),
    ('color=\'cat\' (bin=color)',   p2s.histop(df_h, 'cat', color='cat',   wxh=wxh)),
    ('color=\'group\' (stacked)',   p2s.histop(df_h, 'cat', color='group', wxh=wxh)),
    ('color=\'value\' (spectrum)',  p2s.histop(df_h, 'cat', color='value', wxh=wxh)),
)

### CROW (Color Row) modes — color by raw row count, independent of `count=`

In [ ]:
show_row(
    ('CROW_MAGNITUDEp',              p2s.histop(df_h, 'cat', color=p2s.CROW_MAGNITUDEp, wxh=wxh)),
    ('CROW_STRETCHEDp',              p2s.histop(df_h, 'cat', color=p2s.CROW_STRETCHEDp, wxh=wxh)),
    ('(value, CMAGNITUDE_MEANp)',    p2s.histop(df_h, 'cat', color=('value', p2s.CMAGNITUDE_MEANp), wxh=wxh)),
    ('(value, CSTRETCHED_MEANp)',    p2s.histop(df_h, 'cat', color=('value', p2s.CSTRETCHED_MEANp), wxh=wxh)),
)

### CSETp and CSET_MAGNITUDEp — treat numeric field as categorical

In [ ]:
show_row(
    ('(value, CSETp)',               p2s.histop(df_h, 'cat', color=('value', p2s.CSETp),           wxh=wxh)),
    ('(group, CSET_MAGNITUDEp)',     p2s.histop(df_h, 'cat', color=('group', p2s.CSET_MAGNITUDEp), wxh=wxh)),
    ('(group, CSET_STRETCHEDp)',     p2s.histop(df_h, 'cat', color=('group', p2s.CSET_STRETCHEDp), wxh=wxh)),
    ('color=\'score\' (float)',      p2s.histop(df_h, 'cat', color='score', wxh=wxh)),
)

### color + count combined — color and bar height are independent

In [ ]:
show_row(
    ('color=\'group\', count=\'value\'',
     p2s.histop(df_h, 'cat', color='group', count='value', wxh=wxh)),
    ('color=\'value\', count=\'value\'',
     p2s.histop(df_h, 'cat', color='value', count='value', wxh=wxh)),
    ('color=CROW_MAGNITUDEp, count=\'value\'',
     p2s.histop(df_h, 'cat', color=p2s.CROW_MAGNITUDEp, count='value', wxh=wxh)),
)

---
## Timep — Color Modes
_Source: `test_timep_color.py`_

DataFrame: `ts` (Datetime), `value` (int), `category` ∈ {A,B,C}, `numeric` (float).

In [ ]:
random.seed(42)
df_t = makeTimeDf(n=200, year=(2020, 2024), month=(1, 12))
display(df_t.head())

### Linear mode — same color grid as histop

In [ ]:
wxh_t = (220, 160)
show_row(
    ('color=None',                   p2s.timep(df_t, 'ts', wxh=wxh_t)),
    ('color=\'category\' (stacked)', p2s.timep(df_t, 'ts', color='category', wxh=wxh_t)),
    ('color=\'value\' (spectrum)',   p2s.timep(df_t, 'ts', color='value',    wxh=wxh_t)),
    ('CROW_MAGNITUDEp',              p2s.timep(df_t, 'ts', color=p2s.CROW_MAGNITUDEp, wxh=wxh_t)),
)

### Periodic mode (PT_mp — by month) — same color parameter grid

In [ ]:
show_row(
    ('color=None',                   p2s.timep(df_t, ('ts', p2s.PT_mp), wxh=wxh_t)),
    ('color=\'category\' (stacked)', p2s.timep(df_t, ('ts', p2s.PT_mp), color='category', wxh=wxh_t)),
    ('color=\'value\' (spectrum)',   p2s.timep(df_t, ('ts', p2s.PT_mp), color='value',    wxh=wxh_t)),
    ('CROW_MAGNITUDEp',              p2s.timep(df_t, ('ts', p2s.PT_mp), color=p2s.CROW_MAGNITUDEp, wxh=wxh_t)),
)

### color + count combined (timep)

In [ ]:
show_row(
    ('color=\'category\', count=\'value\'',
     p2s.timep(df_t, 'ts', color='category', count='value', wxh=wxh_t)),
    ('color=\'value\', count=\'numeric\'',
     p2s.timep(df_t, 'ts', color='value', count='numeric', wxh=wxh_t)),
    ('CROW_MAGNITUDEp, count=\'value\'',
     p2s.timep(df_t, 'ts', color=p2s.CROW_MAGNITUDEp, count='value', wxh=wxh_t)),
)

---
## XYp — Color Modes
_Source: `test_xyp_color.py`_

In [ ]:
df_xy = pl.DataFrame({
    'x':     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'y':     [4, 2, 7, 1, 6, 3, 8, 5, 9,  2],
    'group': ['a', 'b', 'a', 'b', 'a', 'b', 'a', 'b', 'a', 'b'],
    'value': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    'size':  [1,  2,  3,  4,  5,  4,  3,  2,  1,  3],
})
display(df_xy)

In [ ]:
wxh_xy = (180, 180)
show_row(
    ('color=None',                   p2s.xyp(df_xy, 'x', 'y', wxh=wxh_xy)),
    ('color=\'group\' (categorical)', p2s.xyp(df_xy, 'x', 'y', color='group', dot_size=4, wxh=wxh_xy)),
    ('color=\'value\' (spectrum)',    p2s.xyp(df_xy, 'x', 'y', color='value', dot_size=4, wxh=wxh_xy)),
    ('CROW_MAGNITUDEp',               p2s.xyp(df_xy, 'x', 'y', color=p2s.CROW_MAGNITUDEp, dot_size=4, wxh=wxh_xy)),
)

In [ ]:
show_row(
    ('CROW_STRETCHEDp',                  p2s.xyp(df_xy, 'x', 'y', color=p2s.CROW_STRETCHEDp, dot_size=4, wxh=wxh_xy)),
    ('(value, CMAGNITUDE_MEANp)',         p2s.xyp(df_xy, 'x', 'y', color=('value', p2s.CMAGNITUDE_MEANp), dot_size=4, wxh=wxh_xy)),
    ('(group, CSET_MAGNITUDEp)',          p2s.xyp(df_xy, 'x', 'y', color=('group', p2s.CSET_MAGNITUDEp), dot_size=4, wxh=wxh_xy)),
    ('(group, CSET_STRETCHEDp)',          p2s.xyp(df_xy, 'x', 'y', color=('group', p2s.CSET_STRETCHEDp), dot_size=4, wxh=wxh_xy)),
)

### XYp multiset sentinel — two overlapping dots with different colors
_Source: `test_xyp_multiset_color.py`_

When two rows land at the same pixel with different categorical colors, the dot gets the multiset sentinel color (`#7f8367`).

In [ ]:
df_overlap = pl.DataFrame({'x': [0.0, 0.0], 'y': [0.0, 0.0], 'c': ['red', 'blue']})
viz_single_red  = p2s.xyp(pl.DataFrame({'x': [0.0], 'y': [0.0], 'c': ['red']}),  'x', 'y', color='c', dot_size=8, wxh=(100, 100))
viz_single_blue = p2s.xyp(pl.DataFrame({'x': [0.0], 'y': [0.0], 'c': ['blue']}), 'x', 'y', color='c', dot_size=8, wxh=(100, 100))
viz_overlap     = p2s.xyp(df_overlap, 'x', 'y', color='c', dot_size=8, wxh=(100, 100))

show_row(
    ('red only',                viz_single_red),
    ('blue only',               viz_single_blue),
    ('red + blue at same pixel → multiset sentinel #7f8367', viz_overlap),
    df=df_overlap,
)

---
## LinkP — node_color Modes
_Source: `test_linkp_color.py`_

Nodes a, b, c, d with edges and category/numeric attributes.

In [ ]:
df_lp = pl.DataFrame({
    'fm':       ['a',     'b',     'c',     'd',     'b'],
    'to':       ['b',     'c',     'd',     'a',     'a'],
    'category': ['cat_x', 'cat_y', 'cat_y', 'cat_x', 'cat_x'],
    'cat_n':    [10,      12,      12,      10,      10],
    'count':    [2.0,     5.0,     10.0,    0.1,     0.5],
})
_pos  = {'a': (0.0, 0.5), 'b': (0.5, 0.0), 'c': (1.0, 0.5), 'd': (0.5, 1.0)}
_base = dict(df=df_lp, relationships=[('fm', 'to')], pos=_pos,
             wxh=(120, 120), link_shape='curve', draw_labels=True, insets=(20, 20))
display(df_lp)

### Hash-based and field-based node coloring

In [ ]:
show_row(
    ('node_color=None',                 p2s.linkp(**_base, node_color=None)),
    ('COLOR_BY_NODE_NAME (hash name)',  p2s.linkp(**_base, node_color=p2s.COLOR_BY_NODE_NAME)),
    ('node_color=\'category\' (CSETp)', p2s.linkp(**_base, node_color='category')),
    ('(category, CSETp)',               p2s.linkp(**_base, node_color=('category', p2s.CSETp))),
)

### Numeric field node coloring — magnitude spectrum

In [ ]:
show_row(
    ('node_color=\'cat_n\' (numeric → magnitude sum)', p2s.linkp(**_base, node_color='cat_n')),
    ('(cat_n, CMAGNITUDE_SUMp)',                        p2s.linkp(**_base, node_color=('cat_n', p2s.CMAGNITUDE_SUMp))),
    ('(cat_n, CSETp) (treat as categorical)',            p2s.linkp(**_base, node_color=('cat_n', p2s.CSETp))),
    ('CROW_MAGNITUDEp (by row count)',                  p2s.linkp(**_base, node_color=p2s.CROW_MAGNITUDEp)),
)

### b and d span two categories → multiset sentinel

In [ ]:
# node 'b': edges fm→c (cat_y) and to←a (cat_x) → two categories → multiset sentinel #7f8367
# node 'd': edges fm→a (cat_x) and to←c (cat_y) → two categories → multiset sentinel
show_row(
    ('node_color=\'category\' — a,c single-cat; b,d multiset sentinel', p2s.linkp(**_base, node_color='category')),
    ('CROW_STRETCHEDp',                                                   p2s.linkp(**_base, node_color=p2s.CROW_STRETCHEDp)),
)

---
## CROW Color Independence
_Source: `test_crow_color_independence.py`_

`CROW_MAGNITUDEp` and `CROW_STRETCHEDp` always color by **raw row count** regardless of what `count=` sizes the bars. The two columns in each pair should have **identical colors** but different bar sizes.

DataFrame: bin `a` has 3 rows (val=1.0 each), `b` has 2 rows (val=5.0), `c` has 1 row (val=10.0).

In [ ]:
df_crow = pl.DataFrame({
    'bin': ['a', 'a', 'a', 'b', 'b', 'c'],
    'val': [1.0, 1.0, 1.0, 5.0, 5.0, 10.0],
})
display(df_crow)

### histop — CROW color is the same whether bars are sized by row count or by `val`

In [ ]:
wxh_c = (180, 150)
show_row(
    # CROW_MAGNITUDEp pair — colors must match, bar heights differ
    ('CROW_MAGNITUDEp, count=row',  p2s.histop(df_crow, 'bin', color=p2s.CROW_MAGNITUDEp,               wxh=wxh_c)),
    ('CROW_MAGNITUDEp, count=\'val\'', p2s.histop(df_crow, 'bin', color=p2s.CROW_MAGNITUDEp, count='val', wxh=wxh_c)),
)
show_row(
    # CROW_STRETCHEDp pair — same
    ('CROW_STRETCHEDp, count=row',  p2s.histop(df_crow, 'bin', color=p2s.CROW_STRETCHEDp,               wxh=wxh_c)),
    ('CROW_STRETCHEDp, count=\'val\'', p2s.histop(df_crow, 'bin', color=p2s.CROW_STRETCHEDp, count='val', wxh=wxh_c)),
)

### timep — same independence

In [ ]:
from datetime import datetime, timedelta
_ts_base = datetime(2024, 1, 1)
df_crow_t = pl.DataFrame({
    'ts':  [_ts_base + timedelta(hours=i) for i in [0, 0, 0, 1, 1, 2]],
    'val': [1.0, 1.0, 1.0, 5.0, 5.0, 10.0],
})
show_row(
    ('CROW_MAGNITUDEp, count=row',     p2s.timep(df_crow_t, 'ts', color=p2s.CROW_MAGNITUDEp,               draw_context=False, wxh=wxh_c)),
    ('CROW_MAGNITUDEp, count=\'val\'', p2s.timep(df_crow_t, 'ts', color=p2s.CROW_MAGNITUDEp, count='val', draw_context=False, wxh=wxh_c)),
    ('CROW_STRETCHEDp, count=row',     p2s.timep(df_crow_t, 'ts', color=p2s.CROW_STRETCHEDp,               draw_context=False, wxh=wxh_c)),
    ('CROW_STRETCHEDp, count=\'val\'', p2s.timep(df_crow_t, 'ts', color=p2s.CROW_STRETCHEDp, count='val', draw_context=False, wxh=wxh_c)),
)

### linkp — same independence (node color by row count, independent of link count)

In [ ]:
df_crow_l = pl.DataFrame({
    'fm':  ['a', 'a', 'a', 'b', 'b', 'c'],
    'to':  ['b', 'b', 'b', 'c', 'c', 'a'],
    'val': [1.0, 1.0, 1.0, 5.0, 5.0, 10.0],
})
_lpos = {'a': (0.0, 0.0), 'b': (1.0, 0.0), 'c': (0.5, 1.0)}
show_row(
    ('CROW_MAGNITUDEp, count=row',     p2s.linkp(df_crow_l, [('fm','to')], _lpos, node_color=p2s.CROW_MAGNITUDEp,               wxh=(120,120), draw_labels=True)),
    ('CROW_MAGNITUDEp, count=\'val\'', p2s.linkp(df_crow_l, [('fm','to')], _lpos, node_color=p2s.CROW_MAGNITUDEp, count='val', wxh=(120,120), draw_labels=True)),
    ('CROW_STRETCHEDp, count=row',     p2s.linkp(df_crow_l, [('fm','to')], _lpos, node_color=p2s.CROW_STRETCHEDp,               wxh=(120,120), draw_labels=True)),
    ('CROW_STRETCHEDp, count=\'val\'', p2s.linkp(df_crow_l, [('fm','to')], _lpos, node_color=p2s.CROW_STRETCHEDp, count='val', wxh=(120,120), draw_labels=True)),
)

---
## Color Overrides
_Source: `test_color_overrides.py`_

`setColorOverrides({'key': '#hex'})` forces specific category values to a given hex color, overriding the default hash.

In [ ]:
df_ov = makeHistoDf(n=200, seed=42)
wxh_ov = (200, 160)

# Baseline — no overrides
p2s.color_overrides_lu.clear()
viz_baseline = p2s.histop(df_ov, 'cat', color='cat', wxh=wxh_ov)

# Override cat 'A' → red
p2s.setColorOverrides({'A': '#ff0000'})
viz_override_one = p2s.histop(df_ov, 'cat', color='cat', wxh=wxh_ov)

# Override 'x' in stacked color
p2s.color_overrides_lu.clear()
p2s.setColorOverrides({'x': '#0066cc'})
viz_override_stacked = p2s.histop(df_ov, 'cat', color='group', wxh=wxh_ov)

p2s.color_overrides_lu.clear()  # always clean up

show_row(
    ('color=\'cat\', no overrides',         viz_baseline),
    ('color=\'cat\', A → #ff0000',          viz_override_one),
    ('color=\'group\' (stacked), x → #0066cc', viz_override_stacked),
)

---
## Histop ↔ Timep Consistency
_Source: `test_bar_colorize_consistency.py`_

For the same DataFrame and `color=` / `count=` parameters, histop (horizontal bars, compare **width**) and timep (vertical bars, compare **height**) must produce identical fill colors. Shown side by side for representative combinations.

In [ ]:
random.seed(42)
# Two-bin DataFrame: two timestamps → two time buckets
from datetime import datetime as _dt
_t1, _t2 = _dt(2025, 5, 10, 12, 0, 0), _dt(2025, 5, 10, 13, 0, 0)
df_cons = pl.DataFrame({
    'ts':        [_t1]*9 + [_t2],
    'int':       [i * 10 for i in range(10)],
    'cat_small': list('AAABBBCCCD'),
})
display(df_cons)

_params = dict(wxh=(160, 160), insets=(2, 2), draw_context=False, distribution=False)

In [ ]:
# For each combo: histop (left) and timep (right) — fills must match
combos = [
    ('color=None',              dict(color=None)),
    ('color=\'cat_small\'',     dict(color='cat_small')),
    ('color=\'int\'',           dict(color='int')),
    ('CROW_MAGNITUDEp',         dict(color=p2s.CROW_MAGNITUDEp)),
    ('(int, CMAGNITUDE_MEANp)', dict(color=('int', p2s.CMAGNITUDE_MEANp))),
    ('count=\'int\', color=\'cat_small\'', dict(count='int', color='cat_small')),
]

for label, extra in combos:
    h = p2s.histop(df_cons, 'ts', **_params, **extra)
    t = p2s.timep( df_cons,       **_params, **extra)
    show_row(
        (f'histop — {label}', h),
        (f'timep — {label}',  t),
    )

---
## Piep — Color Modes
_Source: `test_piep_basic.py`_

Pie/donut/waffle color works the same way as **xyp**: `color=None` outlines each slice in the default data color (fill "none"); pass a field to color by it. To color each slice by its own category, use `color=<bin_by field>`.

DataFrame: `cat` ∈ {A, B, C, D}, `group` ∈ {x, y, z}, `value` (int), `score` (float).

In [ ]:
from piep_dataframes import makePieDf
df_p  = makePieDf(n=200, seed=42)
wxh_p = (150, 150)
display(df_p.head())

# color=None (flat data color) · color by bin category · numeric spectrum · CROW
show_row(
    ('color=None (outline)',               p2s.piep(df_p, 'cat', wxh=wxh_p)),
    ("color='cat' (categorical)",      p2s.piep(df_p, 'cat', color='cat',   wxh=wxh_p)),
    ("color='value' (magnitude sum)",  p2s.piep(df_p, 'cat', color='value', wxh=wxh_p)),
    ('CROW_STRETCHEDp',                p2s.piep(df_p, 'cat', color=p2s.CROW_STRETCHEDp, wxh=wxh_p)),
)

In [ ]:
# CSETp (a different categorical field) · CSET_MAGNITUDEp · fixed hex · cycled hex list
show_row(
    ("color='group' (CSETp)",            p2s.piep(df_p, 'cat', color='group', wxh=wxh_p)),
    ("(group, CSET_MAGNITUDEp)",         p2s.piep(df_p, 'cat', color=('group', p2s.CSET_MAGNITUDEp), wxh=wxh_p)),
    ("color='#4477aa' (fixed)",          p2s.piep(df_p, 'cat', color='#4477aa', wxh=wxh_p)),
    ('hex list (cycled largest-first)',  p2s.piep(df_p, 'cat', color=['#ee6677', '#228833', '#4477aa', '#ccbb44'], wxh=wxh_p)),
)

# The same color= modes apply to donut and waffle styles
show_row(
    ("donut, color='cat'",  p2s.piep(df_p, 'cat', style=p2s.DONUTp,  color='cat', wxh=wxh_p)),
    ("waffle, color='cat'", p2s.piep(df_p, 'cat', style=p2s.WAFFLEp, color='cat', wxh=wxh_p)),
    ("donut, color='value'", p2s.piep(df_p, 'cat', style=p2s.DONUTp, color='value', wxh=wxh_p)),
)

---
## Hex Colors — `#RRGGBBAA` alpha channel (unified detector)
_Source: `test_hex_color_detection.py`_

A single detector (`Polars2SVG.isHexColor` / `isinstance(x, HexColorString)`) now recognizes fixed hex colors identically across every component that accepts one: `#RGB`, `#RRGGBB`, and `#RRGGBBAA` (8-digit, with an **alpha** channel). Before unification, `#ff000080` was a color to linkp/chordp but a "field name" (error) to xyp, and background-shape fills silently dropped anything that wasn't exactly `#RRGGBB`.

_(histop/timep are absent here — they colorize bins and don't accept a fixed `color=` hex.)_

In [ ]:
# Same base color, opaque (#RRGGBB) vs. translucent (#RRGGBBAA). Overlapping xyp dots
# and pie slices make the alpha channel visible.
df_alpha = pl.DataFrame({
    'x': [1.0, 1.5, 1.2, 2.0, 2.0, 2.0, 3.0, 2.7, 2.9],
    'y': [1.0, 1.3, 1.5, 2.0, 2.2, 1.8, 3.0, 2.8, 3.1],
})
show_row(
    ('xyp  #cc3333 (opaque)',   p2s.xyp(df_alpha, 'x', 'y', color='#cc3333',   dot_size=28.0, wxh=(180, 180))),
    ('xyp  #cc333366 (alpha)',  p2s.xyp(df_alpha, 'x', 'y', color='#cc333366', dot_size=28.0, wxh=(180, 180))),
    ('piep #4477aa (opaque)',   p2s.piep(df_p, 'cat', color='#4477aa',   wxh=wxh_p)),
    ('piep #4477aa66 (alpha)',  p2s.piep(df_p, 'cat', color='#4477aa66', wxh=wxh_p)),
)

In [ ]:
# One #RRGGBBAA value renders consistently everywhere a fixed hex color is accepted.
_ALPHA_ = '#33883399'
df_sp = pl.DataFrame({
    'fm':   ['a', 'b', 'c', 'a', 'd', 'b'],
    'to':   ['b', 'a', 'a', 'c', 'a', 'c'],
    'time': [1,   1,   1,   2,   2,   3  ],
})
show_row(
    (f'xyp  {_ALPHA_}',         p2s.xyp(df_xy, 'x', 'y', color=_ALPHA_, dot_size=6, wxh=(160, 160))),
    (f'piep {_ALPHA_}',         p2s.piep(df_p, 'cat', color=_ALPHA_, wxh=wxh_p)),
    (f'linkp {_ALPHA_}',        p2s.linkp(**_base, node_color=_ALPHA_)),
    (f'chordp {_ALPHA_}',       p2s.chordp(df=df_lp, relationships=[('fm', 'to')], node_color=_ALPHA_, wxh=(120, 120))),
    (f'spreadlinesp {_ALPHA_}', p2s.spreadlinesp(df_sp, [('fm', 'to')], ego='a', time='time', node_color=_ALPHA_, wxh=(200, 200))),
)

---
## Legends & Colorbars — `legend=` across components
_Source: `test_legend.py`_

The legend **kind is auto-selected** from the resolved color mode — a categorical
swatch list for `CSETp`/bare-categorical, a colorbar for the spectrum modes
(`CMAGNITUDE_*`, `CSTRETCHED_*`, `CROW_*`, `CSET_MAGNITUDE*`). Swatch colors are
captured from the same hash pipeline the marks use (`component.legend_info` holds
the captured metadata), so legend and marks can never disagree. A truthy
`legend` with nothing to legend (flat/fixed color) silently omits the strip.

In [ ]:
df_leg = makeHistoDf(n=200, seed=42)
wxh_l  = (220, 170)
show_row(
    ('histop categorical (stacked)', p2s.histop(df_leg, 'cat', color='group', legend='bottom', wxh=wxh_l)),
    ('histop colorbar (numeric)',    p2s.histop(df_leg, 'cat', color='value', legend='right',  wxh=wxh_l)),
    ('histop CROW colorbar',         p2s.histop(df_leg, 'cat', color=p2s.CROW_MAGNITUDEp, legend='right', wxh=wxh_l)),
)

In [ ]:
df_xy_leg = pl.DataFrame({
    'x': [1, 2, 3, 4, 5, 6, 7, 8], 'y': [4, 2, 7, 1, 6, 3, 8, 5],
    'cat': ['a', 'b', 'c', 'a', 'b', 'c', 'd', 'e'],
    'val': [1.0, 2.5, 3.2, 0.5, 4.4, 2.2, 1.1, 9.9],
})
show_row(
    ('xyp categorical right', p2s.xyp(df_xy_leg, 'x', 'y', color='cat', dot_size=3.0, legend='right',  wxh=(200, 170))),
    ('xyp colorbar left',     p2s.xyp(df_xy_leg, 'x', 'y', color='val', dot_size=3.0, legend='left',   wxh=(200, 170))),
    ('xyp legend + no color -> silently omitted', p2s.xyp(df_xy_leg, 'x', 'y', dot_size=3.0, legend=True, wxh=(200, 170))),
)

In [ ]:
# piep + linkp: same auto-selection; the swatch hex always equals p2s.color(label)
from piep_dataframes import makePieDf
df_p_leg = makePieDf(n=200, seed=42)
df_lp_leg = pl.DataFrame({
    'fm':   ['a', 'b', 'c', 'a', 'd', 'b', 'e', 'c', 'd', 'e'],
    'to':   ['b', 'c', 'a', 'c', 'a', 'a', 'b', 'd', 'e', 'a'],
    'kind': ['x', 'y', 'x', 'z', 'y', 'x', 'z', 'y', 'x', 'z'],
})
_pos_ = {'a': (0, 0), 'b': (1, 0), 'c': (1, 1), 'd': (0, 1), 'e': (0.5, 0.5)}
show_row(
    ('piep cset',            p2s.piep(df_p_leg, 'cat', color='cat', legend='right', wxh=(220, 150))),
    ('linkp link categorical', p2s.linkp(df_lp_leg, [('fm', 'to')], pos=_pos_, color='kind', legend='right', wxh=(240, 170))),
)
viz = p2s.piep(df_p_leg, 'cat', color='cat', legend='right', wxh=(220, 150))
print('captured metadata:', viz.legend_info)